In [ ]:
# ============================================================
# PARLAY TRAINING - BUDGET MODE
# Target spend: ~$8 per full run on HF Spaces A100
# ============================================================

import os

# --- HF & API Credentials (set as Colab secrets) ---
HF_TOKEN = os.getenv("HF_TOKEN", "")          # HuggingFace token
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")   # Gemini API key
HF_REPO = "your-username/parlay-negotiator"   # CHANGE THIS

# --- Budget Mode Parameters ---
BUDGET_MODE = True   # Set False only if you have >$20 remaining

N_EPISODES = 80 if BUDGET_MODE else 300
GRPO_STEPS = 120 if BUDGET_MODE else 200
GRPO_G = 4 if BUDGET_MODE else 8           # completions per prompt
RUN_SFT = False                             # Skip SFT for budget runs
SFT_STEPS = 100 if not BUDGET_MODE else 0

# --- Model ---
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"
LORA_R = 16
LORA_ALPHA = 32

# --- Env ---
ENV_WS_PORT = 8001
DATA_PATH = "data/episodes.jsonl"
RESULTS_DIR = "results"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)

print(f"Budget mode: {BUDGET_MODE}")
print(f"Episodes: {N_EPISODES}, GRPO steps: {GRPO_STEPS}, G: {GRPO_G}")
print(f"Estimated HF compute cost: ~${N_EPISODES*0.008 + GRPO_STEPS*0.04:.1f}")

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft accelerate bitsandbytes datasets huggingface-hub
!pip install fastapi uvicorn[standard] websockets pydantic aiosqlite
!pip install openenv google-generativeai fastmcp scikit-learn numpy tqdm
print("✓ All dependencies installed")

In [ ]:
import subprocess, sys

# If running in Colab with repo already cloned, skip this cell.
# Otherwise clone:
REPO_URL = "https://github.com/YOUR_USERNAME/parlay.git"  # CHANGE THIS

result = subprocess.run(["git", "clone", REPO_URL, "parlay"], capture_output=True)
if result.returncode == 0:
    %cd parlay
    print("✓ Repository cloned")
else:
    print("Repository already present or clone failed - continuing")
    # Try to cd into it
    try:
        %cd parlay
    except Exception:
        print("Already in repo directory")

In [ ]:
import subprocess, time, threading

def start_env_server():
    proc = subprocess.Popen(
        ["python", "-m", "parlay_env.server", "--port", str(ENV_WS_PORT)],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    return proc

env_proc = start_env_server()
time.sleep(3)  # Give server time to start

# Health check
import websockets, asyncio, json

async def check_env():
    try:
        async with websockets.connect(f"ws://localhost:{ENV_WS_PORT}/ws") as ws:
            await ws.send(json.dumps({"method": "reset",
                                       "params": {"persona": "shark",
                                                  "scenario_id": "saas_enterprise"}}))
            resp = await asyncio.wait_for(ws.recv(), timeout=5)
            data = json.loads(resp)
            print(f"✓ Env server running - observation keys: {list(data.keys())[:5]}")
            return True
    except Exception as e:
        print(f"✗ Env server failed: {e}")
        return False

asyncio.run(check_env())

In [ ]:
# Uses the improved generate_data.py with quality filtering
# See PART 6 of this prompt for the generate_data.py changes

import subprocess

cmd = [
    "python", "training/generate_data.py",
    "--episodes", str(N_EPISODES),
    "--output", DATA_PATH,
    "--quality_filter",
    "--min_efficiency", "0.25",
    "--google_api_key", GOOGLE_API_KEY,
]

print(f"Generating {N_EPISODES} episodes (quality-filtered)...")
result = subprocess.run(cmd, capture_output=False)  # show output live

# Report
import json
records = []
with open(DATA_PATH) as f:
    for line in f:
        records.append(json.loads(line))

deal_rate = sum(1 for r in records if r.get("deal_efficiency", 0) > 0) / len(records)
avg_reward = sum(r.get("reward", 0) for r in records) / len(records)
print(f"\n✓ Generated {len(records)} quality episodes")
print(f"  Deal rate: {deal_rate:.1%}")
print(f"  Avg reward: {avg_reward:.1f}")
print(f"  Train/eval split: {sum(1 for r in records if r.get('split')=='train')} / {sum(1 for r in records if r.get('split')=='eval')}")

In [ ]:
if RUN_SFT:
    print("Running SFT warmup...")
    result = subprocess.run([
        "python", "training/sft_train.py",
        "--data", DATA_PATH,
        "--steps", str(SFT_STEPS),
        "--output", "checkpoints/sft"
    ])
    print("✓ SFT complete" if result.returncode == 0 else "✗ SFT failed")
    SFT_CHECKPOINT = "checkpoints/sft"
else:
    SFT_CHECKPOINT = BASE_MODEL
    print(f"Skipping SFT - starting GRPO from {BASE_MODEL}")

In [ ]:
print("Running random baseline (50 episodes)...")
result = subprocess.run([
    "python", "training/random_baseline.py",
    "--episodes", "50",
    "--output", f"{RESULTS_DIR}/random_baseline.json"
])

import json
with open(f"{RESULTS_DIR}/random_baseline.json") as f:
    baseline = json.load(f)

RANDOM_BASELINE_REWARD = baseline["mean_reward"]
print(f"✓ Random baseline mean reward: {RANDOM_BASELINE_REWARD:.1f}")

In [ ]:
print(f"Starting GRPO training: {GRPO_STEPS} steps, G={GRPO_G}...")
print("Training loop connects to live env WS - not replaying JSONL")

result = subprocess.run([
    "python", "training/grpo_train.py",
    "--base_model", SFT_CHECKPOINT,
    "--steps", str(GRPO_STEPS),
    "--g", str(GRPO_G),
    "--env_port", str(ENV_WS_PORT),
    "--output", "checkpoints/grpo",
    "--save_curves", f"{RESULTS_DIR}/grpo_curves.json",
])

print("✓ GRPO complete" if result.returncode == 0 else "✗ GRPO failed - check output above")

In [ ]:
import json, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Load GRPO curves
with open(f"{RESULTS_DIR}/grpo_curves.json") as f:
    curves = json.load(f)

# Run evaluation against live env
result = subprocess.run([
    "python", "training/evaluate.py",
    "--base_model", BASE_MODEL,
    "--sft_checkpoint", "checkpoints/sft" if RUN_SFT else "",
    "--grpo_checkpoint", "checkpoints/grpo",
    "--env_port", str(ENV_WS_PORT),
    "--output", f"{RESULTS_DIR}/eval_results.json",
    "--save_transcript",
])

with open(f"{RESULTS_DIR}/eval_results.json") as f:
    eval_results = json.load(f)

# -- Plot 1: 4-bar comparison chart --
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#1c2b1a')
ax.set_facecolor('#1c2b1a')

labels = ["Random\n(baseline)", "Base Model\n(Qwen2.5-1.5B)", "GRPO Trained\n(Ours)"]
values = [
    RANDOM_BASELINE_REWARD,
    eval_results.get("base_mean_reward", 0),
    eval_results.get("grpo_mean_reward", 0),
]
if RUN_SFT and "sft_mean_reward" in eval_results:
    labels.insert(2, "SFT Warmup")
    values.insert(2, eval_results["sft_mean_reward"])

colors = ['#8b1a1a', '#8a8070', '#4a7c59', '#c9a84c']
bars = ax.bar(labels, values, color=colors[:len(labels)], width=0.5, edgecolor='#c9a84c', linewidth=0.8)

ax.axhline(0, color='#c9a84c', linewidth=0.8, linestyle='--', alpha=0.5, label='Zero reward')
ax.set_xlabel("Agent Type", color='#f5f0e8', fontsize=11)
ax.set_ylabel("Mean Episode Reward", color='#f5f0e8', fontsize=11)
ax.set_title("Parlay - Negotiation Agent Training Progress", color='#c9a84c', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(colors='#f5f0e8')
for spine in ax.spines.values():
    spine.set_color('#c9a84c')
    spine.set_alpha(0.4)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f"{val:+.0f}", ha='center', va='bottom', color='#f5f0e8', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/training_curves.png", dpi=150, bbox_inches='tight', facecolor='#1c2b1a')
plt.show()
print(f"✓ Saved: {RESULTS_DIR}/training_curves.png")

# -- Plot 2: GRPO reward curve over training steps --
if "step_rewards" in curves:
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    fig2.patch.set_facecolor('#1c2b1a')
    ax2.set_facecolor('#1c2b1a')

    steps = list(range(len(curves["step_rewards"])))
    rewards = curves["step_rewards"]
    window = max(1, len(rewards) // 20)
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')

    ax2.plot(steps[:len(smoothed)], smoothed, color='#c9a84c', linewidth=2, label='Mean episode reward (smoothed)')
    ax2.fill_between(steps[:len(smoothed)], smoothed, alpha=0.2, color='#c9a84c')
    ax2.axhline(RANDOM_BASELINE_REWARD, color='#8b1a1a', linewidth=1, linestyle='--', label=f'Random baseline ({RANDOM_BASELINE_REWARD:.0f})')
    ax2.set_xlabel("Training Step", color='#f5f0e8', fontsize=11)
    ax2.set_ylabel("Mean Episode Reward", color='#f5f0e8', fontsize=11)
    ax2.set_title("Parlay GRPO - Reward Curve", color='#c9a84c', fontsize=13, pad=12)
    ax2.tick_params(colors='#f5f0e8')
    ax2.legend(facecolor='#2c1810', edgecolor='#c9a84c', labelcolor='#f5f0e8')
    for spine in ax2.spines.values():
        spine.set_color('#c9a84c')
        spine.set_alpha(0.4)

    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/grpo_reward_curve.png", dpi=150, bbox_inches='tight', facecolor='#1c2b1a')
    plt.show()
    print(f"✓ Saved: {RESULTS_DIR}/grpo_reward_curve.png")

In [ ]:
import json

transcript_path = f"{RESULTS_DIR}/before_after_transcript.json"
if os.path.exists(transcript_path):
    with open(transcript_path) as f:
        transcripts = json.load(f)

    html = """
<html><head>
<style>
body { background: #1c2b1a; color: #f5f0e8; font-family: 'EB Garamond', serif;
       margin: 0; padding: 20px; }
h1 { color: #c9a84c; font-family: 'Playfair Display', serif; text-align: center; }
.cols { display: flex; gap: 20px; }
.col { flex: 1; border: 1px solid #c9a84c; padding: 15px; background: #2c1810; }
.col h2 { color: #c9a84c; font-size: 1.1rem; margin-top: 0; }
.turn { margin: 8px 0; padding: 8px; border-radius: 4px; font-size: 0.9rem; }
.player { background: #2a3d28; border-left: 3px solid #c9a84c; }
.agent { background: #1a1208; border-left: 3px solid #8b1a1a; }
.annotation { color: #c9a84c; font-size: 0.75rem; font-style: italic; margin: 2px 0; }
.bad { border-left-color: #8b1a1a !important; background: #2c1010 !important; }
.good { border-left-color: #1a5c2a !important; background: #102c10 !important; }
</style></head><body>
<h1>◈ PARLAY - Before vs After Training</h1>
<div class="cols">
"""
    for label, key in [("Base Model (Step 0)", "base"), ("GRPO Trained (Step 120)", "grpo")]:
        html += f'<div class="col"><h2>{label}</h2>'
        turns = transcripts.get(key, {}).get("turns", [])
        for t in turns:
            role_class = "player" if t["role"] == "player" else "agent"
            annotation = t.get("annotation", "")
            bad = " bad" if t.get("is_bad") else ""
            good = " good" if t.get("is_good") else ""
            html += f'<div class="turn {role_class}{bad}{good}">'
            html += f'<strong>{t["role"].upper()}:</strong> {t["text"]}'
            if annotation:
                html += f'<div class="annotation">▸ {annotation}</div>'
            html += '</div>'
        total_reward = transcripts.get(key, {}).get("total_reward", "N/A")
        html += f'<div style="margin-top:10px;color:#c9a84c;font-size:0.9rem;">Total reward: <strong>{total_reward}</strong></div>'
        html += '</div>'

    html += "</div></body></html>"

    with open(f"{RESULTS_DIR}/before_after_transcript.html", "w") as f:
        f.write(html)

    from IPython.display import HTML
    display(HTML(html))
    print(f"✓ Saved: {RESULTS_DIR}/before_after_transcript.html")
else:
    print("No transcript file found - run evaluate.py with --save_transcript flag")

In [ ]:
import subprocess

# Commit all result files to repo
subprocess.run(["git", "add", "results/"])
subprocess.run(["git", "commit", "-m",
    "results: add training curves, eval comparison, before/after transcript"])
subprocess.run(["git", "push"])

# Push trained model to HF Hub
if HF_TOKEN:
    result = subprocess.run([
        "python", "training/push_to_hub.py",
        "--checkpoint", "checkpoints/grpo",
        "--repo", HF_REPO,
        "--token", HF_TOKEN,
    ])
    print("✓ Model pushed to HF Hub" if result.returncode == 0 else "✗ Push failed")
else:
    print("⚠ No HF_TOKEN set - skipping model push")

print("\n=== TRAINING COMPLETE ===")
print(f"Results saved to: {RESULTS_DIR}/")
print("Files to commit to README:")
for f in ["training_curves.png", "grpo_reward_curve.png", "before_after_transcript.html"]:
    import os
    path = f"{RESULTS_DIR}/{f}"
    exists = "✓" if os.path.exists(path) else "✗"
    print(f"  {exists} {path}")